# Phase 4 — Independent validations

**4.1 CFTP (Corcoran–Tweedie perfect sampling).** For an IMH chain, all coupled
chains coalesce the first step the worst-case config `s★ = argmax_x w(x)` is
proposed (prob `q(s★) = π(s★)/C` per step), so the mean coalescence time is
`1/q(s★) = C/π(s★)`. Measuring it independently re-derives `C`.

**4.2 Speedup vs local dynamics.** Single-spin-flip Metropolis (Glauber) `τ_int`
at the same `(L,β)` quantifies why collective updates matter. *(Caveat: τ_int is in
units of each method's "sweep" — a TNMH step contracts the whole lattice, a Glauber
sweep is N flips; this compares decorrelation per sweep, not wall-clock.)*


In [ ]:
using Plots, JLD2, Statistics, LinearAlgebra, DelimitedFiles, Printf
include("main.jl"); include("tools/tnmh_tools.jl")
mkpath("results")
set_seed(20240618)
betac = log(1 + sqrt(2)) / 2          # β_c = ln(1+√2)/2 ≈ 0.4407
log_finding(s) = open(io -> println(io, s), "results/FINDINGS.md", "a")
println("ready. β_c = ", round(betac, digits=6))


In [ ]:
function imh_energy_series(Lx, Ly, beta, D, N; J=1.0)
    cfg, lq = sample_config_opt(Lx, Ly, beta, D, J); e = measure_energy(cfg, J)
    Es = Float64[]
    for _ in 1:N
        ncfg, nlq = sample_config_opt(Lx, Ly, beta, D, J); ne = measure_energy(ncfg, J)
        if log(rand()) < (lq - nlq) - beta * (ne - e); cfg, lq, e = ncfg, nlq, ne; end
        push!(Es, e)
    end
    return Es
end
println("helper ready")


## 4.1 — CFTP coalescence time vs `C/π(s★)`

In [ ]:
cftp_cases = [(3,4,2), (2,5,2)]
emp = Float64[]; pred = Float64[]; Cs = Float64[]
for (Lx, Ly, D) in cftp_cases
    cf = cftp_coalescence(Lx, Ly, betac, D; ntrials=300)
    push!(emp, cf.mean_time); push!(pred, cf.predicted); push!(Cs, cf.C)
    @printf("Lx=%d Ly=%d D=%d | mean coalescence = %.2f   predicted 1/q(s★) = %.2f   C = %.5f\n",
            Lx, Ly, D, cf.mean_time, cf.predicted, cf.C)
end


## 4.2 — TNMH vs single-spin-flip Glauber

In [ ]:
L = 8
Es_tn = imh_energy_series(L, L, betac, 2, 3000)
tau_tn = integrated_autocorr(Es_tn[601:end])
g = glauber_tau_int(L, L, betac, 3000)
tau_loc = g.tau_int
@printf("L=%d, β_c:  τ_int(TNMH)=%.2f   τ_int(Glauber)=%.2f   ratio=%.1f×\n",
        L, tau_tn, tau_loc, tau_loc / tau_tn)


## Save + record finding

In [ ]:
jldsave("results/phase4_validation.jld2";
        cftp_cases=string.(cftp_cases), cftp_emp=emp, cftp_pred=pred, cftp_C=Cs,
        tau_tn=tau_tn, tau_glauber=tau_loc)
log_finding("\n## Phase 4 — Validations")
log_finding("- CFTP mean coalescence ≈ predicted 1/q(s★)=C/π(s★) (re-derives C independently).")
log_finding("- TNMH τ_int=$(round(tau_tn,digits=2)) vs Glauber τ_int=$(round(tau_loc,digits=2)) at L=$L (ratio $(round(tau_loc/tau_tn,digits=1))×, per-sweep).")
println("saved results/phase4_validation.jld2")
